# Spectral-tilt response: EE2, HMcode2020 and SYREN-NEW

Why can nonlinear spectra agree to a few percent while their photometric Fisher forecasts differ much more? This experiment isolates the **fixed-As response to ns** in linear and nonlinear total-matter power.

**Main finding:** at low redshift, an explicitly ns-dependent term in the symbolic nonlinear formula largely cancels the primordial tilt response. Fixing ns in the saved Fisher matrices reduces several marginalized-error discrepancies. This supports an important ns-degeneracy contribution, not a claim that ns explains every difference.

- [Full report](../docs/source/syren_ns_response.md)
- [Reusable calculation and plotting script](../scripts/validation/plot_ns_power_response.py)
- [Original photometric notebook](SYREN-investigation_Photo-EE2-vs-matched.ipynb)

**Run All reads the supplied artifacts by default.** Only enabling `RECOMPUTE` launches the nine cosmology evaluations (roughly 1–3 minutes, environment-dependent). Cached analysis needs NumPy, Matplotlib and IPython; recomputation also needs this checkout and CLASS, EuclidEmulator2 and symbolic_pofk in the kernel environment.

## 1. Comparison contract

Fiducial: `10^9As=2.1, Omegam=0.32, Omegab=0.05, h=0.67, ns=0.96, mnu=0.06 eV`; fixed `w0=-1, wa=0`. Use `class/ee2_boost_photo.yaml`, `class/hmcode2020_photo.yaml`, and `symbolic/syren_new_photo.yaml`, the profiles used by the successful photometric notebook.

For each backend, evaluate the fiducial and ns±0.0096 cosmologies. All other parameters, especially **As rather than sigma8**, remain fixed. Plot k=0.01–3 h/Mpc at z=0,1,2.

$$R_{n_s}(k,z)=\frac{\partial\ln P(k,z)}{\partial n_s}\simeq\frac{\ln P(n_s+\Delta n_s)-\ln P(n_s-\Delta n_s)}{2\Delta n_s}.$$

The script calls `cosmo_functions.Pmm(z, k*h, nonlinear=...)` for all backends and converts Mpc³ to (Mpc/h)³. This tests the actual wrapper and interpolation path. It rejects non-positive/non-finite powers and evaluation outside the source spline support.

Lower response panels show **additive differences from CLASS+EE2**. Ratios of derivatives can diverge at zero crossings, so they are not the default diagnostic. EE2 is a comparison reference, not exact ground truth.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import numpy as np
from IPython.display import Image, Markdown, display

repo_root = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'scripts/validation/plot_ns_power_response.py').is_file()
)
script = repo_root / 'scripts/validation/plot_ns_power_response.py'
response_dir = repo_root / 'docs/source/_static/ns_response'
RECOMPUTE = False
NS_STEP = 0.0096  # absolute half-step, only used when RECOMPUTE=True
COSMO_MODEL = 'LCDM'

def markdown_table(headers, rows):
    lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    lines += ['| ' + ' | '.join(str(value) for value in row) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))


### Optional recomputation

Set `RECOMPUTE=True` above to invoke the shared script using this kernel's Python environment. New runs go to `results/ns_response_notebook`, preserving the documentation snapshot. Change `NS_STEP` to investigate finite-difference sensitivity. Nine spectra are computed; no Fisher forecast runs.

For command-line use: `OMP_NUM_THREADS=1 uv run python scripts/validation/plot_ns_power_response.py`. Its `--plot-only` option redraws existing arrays without invoking solvers.

In [ ]:
if RECOMPUTE:
    response_dir = repo_root / 'results/ns_response_notebook'
    command = [sys.executable, str(script), '--ns-step', str(NS_STEP),
               '--cosmo-model', COSMO_MODEL, '--output-dir', str(response_dir)]
    subprocess.run(command, cwd=repo_root, env={**os.environ, 'OMP_NUM_THREADS': '1'}, check=True)
else:
    print('Using saved response artifacts; no cosmology calculations launched.')


In [ ]:
archive = response_dir / 'ns_response.npz'
if not archive.is_file():
    raise FileNotFoundError(f'{archive}: enable RECOMPUTE or run the linked script first.')
with np.load(archive, allow_pickle=False) as saved:
    data = {key: saved[key] for key in saved.files}
metadata = json.loads((response_dir / 'ns_response_metadata.json').read_text())
k, zs = data['k_hmpc'], data['redshifts']
labels = data['labels'].tolist()
response, power = data['response'], data['power']
print('Saved model:', str(data['cosmo_model']), '| saved ns half-step:', float(data['ns_step']))
print('Created:', metadata['created_utc'])
print('Fiducial:', metadata['fiducial'])
print('Power axes:', metadata['power_axes'], '| shape:', power.shape)
print('Power units:', metadata['power_units'])
markdown_table(['Backend', 'Profile'], [
    (label, Path(metadata['profiles'][label]['path']).name) for label in labels
])


## 2. Linear response: the normalization check

At fixed As, the primordial factor is `(k*h/0.05)**(ns-1)`, so the exact linear tilt response is **ln(k*h/0.05)**. The remaining direct symbolic linear factors on this path do not depend on ns. The wrapper supplies ns in the correct argument position and does not invoke sigma8-to-As conversion when As is provided.

All three responses should closely track this curve. Small public-interface residuals can arise from interpolating power on finite grids; the raw symbolic expression satisfies the analytic identity to numerical precision.

In [ ]:
display(Image(filename=str(response_dir / 'ns_response_linear.png'), width=1200))
exact_linear = np.log(k * float(data['h']) / 0.05)
markdown_table(['Backend', 'max |Rlin − exact tilt| on saved grid'], [
    (label, f'{np.max(np.abs(response[i, 0] - exact_linear)):.6g}')
    for i, label in enumerate(labels)
])


## 3. Nonlinear response: where the backends differ

The initial direct-formula analysis found a broad low-z SYREN response deficit, especially near k=0.2–1 h/Mpc. The three-backend public-interface run confirms it. At z=0, k=0.5 the saved responses are approximately **EE2=1.071, HMcode=1.023, SYREN=0.443**.

**Additional observation:** HMcode departs substantially at high k and z=2 in this run. The cause is unresolved; step-size and direct CLASS-accessor checks are needed before attributing it to physics or interpolation. This is distinct from the SYREN formula diagnosis.

In [ ]:
display(Image(filename=str(response_dir / 'ns_response_nonlinear.png'), width=1200))
rows = []
for j, z in enumerate(zs):
    for anchor in (0.2, 0.5, 1.0):
        matches = np.flatnonzero(np.isclose(k, anchor, rtol=1e-12))
        if matches.size:
            rows.append([f'{z:g}', f'{anchor:g}'] +
                        [f'{value:.6f}' for value in response[:, 1, j, matches[0]]])
markdown_table(['z', 'k [h/Mpc]', *labels], rows)


## 4. Remove the common primordial tilt

The response of the nonlinear boost is **R(Pnl/Plin) = R(Pnl) − R(Plin)**. This isolates the parameter dependence of nonlinear corrections from the common linear tilt.

In [ ]:
display(Image(filename=str(response_dir / 'ns_response_boost.png'), width=1200))


## 5. Accurate values do not guarantee accurate derivatives

The fiducial spectra are much closer than some of the parameter responses. If `P_syren = P_ref * exp(epsilon)`, then `R_syren − R_ref = d epsilon / d ns`. A small value error at the fiducial does not constrain the slope of that error.

The following residuals are for the fiducial power values; unlike response ratios, they have a strictly positive denominator.

In [ ]:
display(Image(filename=str(response_dir / 'ns_fiducial_spectra.png'), width=1200))


## 6. The sensitive symbolic term

Upstream `symbolic_pofk.syren_new.pnl_new_emulated` uses `L=log10(Plin)` and constructs `log10(Pnl)=L+term2+term3−term4−term5−bias(k)`. In particular:

$$T_3=\frac{(0.6264a-0.3035L+0.6069)k}{0.7882\Omega_m+0.4811k+1.4326n_s-1.8971+(0.0271L+0.9635k^{0.0264})^{22.9213a-71.1658n_s}}.$$

At a=1 and ns=0.96 its exponent is **−45.397868**. `term2` has another explicit exponent `27.6818*a−24.8736*ns`. Large coefficients alone do not imply a bug; the diagnostic is their differentiated contribution.

The previously measured chain-rule decomposition at **z=0, k=0.5 h/Mpc** is:

| Contribution to d ln Pnl / d ns | Value |
|---|---:|
| Linear tilt | +1.90210753 |
| term2 via L | −0.13125580 |
| term2 explicit ns | +0.01353874 |
| term3 via L | +0.57108368 |
| term3 explicit ns | **−1.91242415** |
| Total | **+0.44304999** |

The explicit term3 response almost cancels the linear tilt. The ns-independent `pnl_bias(k)` cannot alter the logarithmic response at fixed h. Do not delete or retune these calibrated terms based on this diagnostic alone. The table is historical direct-formula evidence; this notebook does not duplicate the symbolic expression.

## 7. Step-size evidence and its scope

In the direct symbolic formula, over k=[0.03,0.05,0.1,0.2,0.5,1,3] h/Mpc and z=[0,1,2], decreasing the half-step from 0.0096 to 0.0001 changed the log response by at most **0.00078737**; 0.001 to 0.0001 changed it by at most **8.459e−6**. This is much smaller than the low-z mismatch.

That test was of the direct formula, not a convergence certification of CLASS, HMcode or every projected derivative. At step=0.0001 and z=0, k=0.5, the direct comparison gave **SYREN=0.443050 versus EE2 boost + exact linear tilt=1.074483**. The public-interface run uses spline interpolation and a larger step, explaining why its values are not identical.

## 8. Inspect a redshift or the absolute power derivative

The arrays also contain `dP_dns=(Pplus−Pminus)/(2*step)`, in power units. At finite step it is not identical to `Pfid*R`. Change the two selections below to inspect either statistic without rerunning the backends.

In [ ]:
import matplotlib.pyplot as plt

Z_INDEX = 0
QUANTITY = 'response'  # or 'dP_dns' for the absolute derivative
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for t, title in enumerate(('Linear', 'Nonlinear')):
    for i, label in enumerate(labels):
        axes[t].semilogx(k, data[QUANTITY][i, t, Z_INDEX],
                         color=('#0072B2', '#009E73', '#D55E00')[i],
                         ls=('-', '-.', '--')[i], label=label)
    axes[t].set(title=f'{title}, z={zs[Z_INDEX]:g}', xlabel='k [h/Mpc]', ylabel=QUANTITY)
    axes[t].grid(alpha=0.2)
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()


## 9. Connection to the photometric Fisher results

The following is a snapshot of the previously inspected **saved** Fisher matrices, not a Fisher calculation performed by this notebook. Fixing ns deletes its row and column from the full Fisher matrix before inversion; all other retained parameters, including nuisances, are marginalized.

| LCDM marginalized-error difference, SYREN vs EE2 | ns free | ns fixed |
|---|---:|---:|
| As | −34.73% | **−4.84%** |
| Omegam | −3.82% | −1.52% |
| Omegab | −21.97% | −8.22% |
| h | −33.76% | **−7.37%** |

A weaker individual derivative can nevertheless yield tighter marginalized constraints because the derivative **shape** changes degeneracies. SYREN's full conditional LCDM ns error is 8.53% larger than EE2's, but its marginalized ns error is 33.78% smaller.

Adding w0 and wa weakens absolute errors, while relative backend differences can shrink. Saved marginalized sigma(ns) values:

| Model | EE2 | HMcode2020 | SYREN-NEW |
|---|---:|---:|---:|
| LCDM | 0.015057 | 0.015340 | 0.009971 |
| w0waCDM | 0.015418 | 0.019250 | 0.013607 |

**Original notebook caveats:** `compare_to_index=1` selects HMcode despite an EE2 axis label; index 0 selects EE2. The darker 'unmarginalized' shading is calculated after omitted nuisance parameters have been marginalized. Its ns difference is approximately 10.27%, not the full-conditional 8.53%. These response figures explicitly use EE2 throughout.

## 10. Conclusions and next checks

1. Linear ns responses agree closely: amplitude normalization or incorrect argument ordering is not supported as the cause.
2. SYREN's low-z nonlinear response mismatch is present both in the direct formula and in the public-interface comparison.
3. Fixing ns substantially reduces several saved LCDM error differences, supporting an important ns-degeneracy contribution.
4. The new z=2 HMcode feature requires separate step-size and direct-accessor investigation.
5. To quantify causal Fisher impact, swap **only the ns angular-spectrum derivative** while keeping covariance, every other derivative, nuisance treatment and priors common. This controlled experiment has not yet been run.

The linked report preserves the detailed evidence and qualifications; `ns_response_metadata.json` records the exact profiles, source hashes, dependency versions, grid and step for the displayed run.